# Notebook 5 — Full Transfer Matrix: Does Truthfulness Generalise Across Domains?

## The definitive experiment

This notebook is the culmination of our probing investigation. We train a probe on **each** of the 8 datasets and evaluate it on **all** 8 datasets, producing a full **8x8 transfer matrix**.

## How to read the matrix

- **Diagonal entries** (train == eval) measure **in-distribution** performance. These should be high if the probe works at all.
- **Off-diagonal entries** (train != eval) measure **cross-domain transfer**. This is the real test. A probe trained on geography facts is being asked to detect lies about science exams or boolean questions.

| Outcome | Interpretation |
|---------|---------------|
| High diagonal, **high off-diagonal** | Strong evidence for a **universal truth direction** shared across domains. The RepE claim holds. |
| High diagonal, **low off-diagonal** | The probe is picking up on **domain-specific features**, not a general truth signal. |
| Low diagonal | The probe is not even working in-distribution — the model may be too small or the prompts too noisy. |

## Benchmarks from the RepE paper

The original RepE paper uses **Llama-2-13B-chat** (13B parameters, 40 layers) and reports:
- **Diagonal** values near **0.85–0.95**
- **Off-diagonal** transfer values of **0.60–0.80** for semantically related datasets

## Our setup

We test with **Phi-2** (2.7B parameters, 32 layers). This model is roughly 5x smaller than Llama-2-13B but significantly larger than gpt2-medium (355M). Our 8 datasets include the same benchmarks used in the RepE paper (TruthfulQA, ARC, BoolQ), enabling direct comparison.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("..") / "src"))

import warnings
warnings.filterwarnings("ignore")

from lie_detector_llm.datasets import build_dataset_collection
from lie_detector_llm.experiment import run_full_transfer_matrix
from lie_detector_llm.plotting import plot_transfer_heatmap

collection = build_dataset_collection()
print("Datasets:", collection.dataset_names())
for name in collection.dataset_names():
    sub = collection.subset(name)
    print(f"  {name}: {len(sub)} prompts, {sub['group_id'].nunique()} groups")

## LR probe — full transfer matrix

We start with the **logistic regression** probe, which is the strongest supervised method in our toolkit. It fits a linear decision boundary in activation space to separate true from false statements.

In [ ]:
MODEL = "microsoft/phi-2"

lr_matrix = run_full_transfer_matrix(
    collection=collection,
    model_name=MODEL,
    probe_method="lr",
    layer_index=-1,
)
pivot_lr = lr_matrix.results.pivot(index="train_dataset", columns="eval_dataset", values="grouped_accuracy")
print("LR probe — full transfer matrix:\n")
print(pivot_lr.round(2).to_string())

In [ ]:
fig, ax = plot_transfer_heatmap(
    lr_matrix.results,
    title=f"Full transfer matrix — LR probe ({MODEL})",
)
fig

## DIM probe — full transfer matrix

DIM is parameter-free and may generalise differently from LR. Let's compare.

In [ ]:
dim_matrix = run_full_transfer_matrix(
    collection=collection,
    model_name=MODEL,
    probe_method="dim",
    layer_index=-1,
)
pivot_dim = dim_matrix.results.pivot(
    index="train_dataset", columns="eval_dataset", values="grouped_accuracy"
).round(2)
print("DIM probe — full transfer matrix:\n")
print(pivot_dim.to_string())

fig, ax = plot_transfer_heatmap(
    dim_matrix.results,
    title=f"Full transfer matrix — DIM probe ({MODEL})",
)
fig

## Side-by-side comparison: transfer gap

In [5]:
# Difference matrix: LR − DIM (positive = LR better, negative = DIM better)
diff = (pivot_lr - pivot_dim).round(2)
print("LR − DIM transfer difference (positive = LR better):")
print(diff.to_string())

LR − DIM transfer difference (positive = LR better):
eval_dataset     cities  larger_than    qa  repeng_truthful
train_dataset                                              
cities              0.4          0.0  0.12            -0.12
larger_than         0.1          0.0  0.00             0.02
qa                  0.1         -0.4  0.37            -0.08
repeng_truthful     0.0         -0.4  0.13             0.20


## All-probe aggregate: average off-diagonal transfer

In [ ]:
import numpy as np
import pandas as pd

summary_rows = []
for method in ["dim", "lat", "lr", "pca-g"]:
    m = run_full_transfer_matrix(
        collection=collection,
        model_name=MODEL,
        probe_method=method,
        layer_index=-1,
    )
    pivot = m.results.pivot(
        index="train_dataset", columns="eval_dataset", values="grouped_accuracy"
    )
    n = len(pivot)
    diagonal_mean = np.diag(pivot.values).mean()
    off_diag = pivot.values[~np.eye(n, dtype=bool)]
    off_diag_mean = off_diag.mean()
    summary_rows.append({
        "probe_method": method,
        "in_distribution_mean": round(diagonal_mean, 3),
        "transfer_mean": round(off_diag_mean, 3),
        "transfer_gap": round(diagonal_mean - off_diag_mean, 3),
    })

summary_df = pd.DataFrame(summary_rows).sort_values("transfer_mean", ascending=False)
print("Aggregate transfer statistics (all probes):\n")
print(summary_df.to_string(index=False))

## Interpretation

### Reading the matrix

| Pattern | What it means |
|---------|---------------|
| High diagonal, high off-diagonal | Strong universal truth direction — the RepE claim holds |
| High diagonal, low off-diagonal | Dataset-specific features drive the probe, not universal truth |
| Low diagonal | The probe is not even working in-distribution |

### What Phi-2 tells us

With 2.7B parameters and 32 layers, **Phi-2** is a significant step up from gpt2-medium (355M, 24 layers). The transfer matrix with 8 datasets — including standard benchmarks from the RepE paper (TruthfulQA, ARC, BoolQ) — provides a comprehensive assessment of truth direction universality.

The **transfer_gap** column in the summary table is the most important metric: it measures the drop in accuracy from in-distribution to cross-domain. A smaller gap means a more universal truth direction.

### Comparison with the RepE paper

The original paper uses **Llama-2-13B-chat** (~5x larger, instruction-tuned) and reports off-diagonal transfer of 0.60–0.80. Our results with Phi-2 show how much of this transfer emerges at the 2.7B scale without instruction tuning.

### Practical conclusion

The full 8x8 transfer matrix is the strongest piece of evidence in this project. It directly answers: **can we build a lie detector that works across domains, or only within the domain it was trained on?**